In [1]:
import os
from dotenv import load_dotenv
from nltk.data import retrieve

# from data_ingest_parsing.data_ingestion import documents

load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_core.documents import Document

## vector store
from langchain_community.vectorstores import Chroma

import numpy as np
from typing import List

F:\rag_practice_repo\rag_practice_repo\DATA_PARSING_INGESTION\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



In [4]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning finds patterns in unlabeled data. Reinforcement learning learns through
    interaction with an environment using rewards and penalties.
    """,

    """
    Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning has revolutionized fields like computer vision, natural language
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers
    excel at sequential data processing.
    """,

    """
    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language.
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis,
    machine translation, and question answering. Modern NLP heavily relies on transformer
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand
    context and relationships between words in text.
    """
]

sample_docs

['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning finds patterns in unlabeled data. Reinforcement learning learns through\n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image 

In [5]:
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt",'w') as f:
        f.write(doc)

print(f"sample doc created in : {temp_dir}")

sample doc created in : C:\Users\asus\AppData\Local\Temp\tmpfbtq5ud6


In [44]:
# doc loading
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)
documents=loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\n First doc preview")
print(documents)

Loaded 3 documents

 First doc preview
[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    '), Document(metadata={'source': 'data\\doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has rev

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Document Splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

In [8]:
chunks = text_splitter.split_documents(documents)

In [9]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of laye

In [10]:
embeddings=OpenAIEmbeddings()
sample_text="Machine learning is a subset of artificial intelligence that enables system"

In [11]:
vector=embeddings.embed_query(sample_text)
vector

[-0.024885298684239388,
 -0.00548783503472805,
 -0.013962972909212112,
 -0.0027622547931969166,
 -0.014683126471936703,
 0.03408725932240486,
 -0.006854793056845665,
 -0.004967724438756704,
 -0.02061772346496582,
 -0.021644609048962593,
 0.020044267177581787,
 0.015870045870542526,
 -0.0031256654765456915,
 -0.010828972794115543,
 -0.0008635171689093113,
 0.004797688219696283,
 0.009768746793270111,
 0.025218702852725983,
 0.014843160286545753,
 -0.011182380840182304,
 -0.01816386729478836,
 0.018670642748475075,
 0.007641626987606287,
 -0.023058243095874786,
 -0.0013844614150002599,
 -0.02499198727309704,
 0.008861887268722057,
 -0.04166220501065254,
 -0.0067180972546339035,
 0.010042138397693634,
 0.013309500180184841,
 -0.014269704930484295,
 -0.02019096538424492,
 -0.026659009978175163,
 -0.0249653160572052,
 -0.014483083970844746,
 -0.001228594919666648,
 0.0024005111772567034,
 0.014056325890123844,
 -0.001730368472635746,
 0.021257858723402023,
 0.01801716908812523,
 -0.00134445

### CHROMA DB

In [12]:
persist_dir="./chroma_db"
# initialize chromadb with openai embeddings
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(),
    persist_directory=persist_dir,
    collection_name="rag_collection",
)
print(vectorstore._collection.count())
print(persist_dir)

15
./chroma_db


### Test similarity Search

In [13]:
query="what is nlp"
similar_docs=vectorstore.similarity_search(query,k=1)
similar_docs

[Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.')]

In [14]:
### Advanced similarity Search with score
vectorstore.similarity_search_with_score(query,k=3)

[(Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
  0.30609583854675293),
 (Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and 

In [15]:
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(
    model_name="gpt-3.5-turbo"
)


In [16]:
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000021085DF38E0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000021085E1A170>, root_client=<openai.OpenAI object at 0x00000210F0C3BC70>, root_async_client=<openai.AsyncOpenAI object at 0x0000021085DF30D0>, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [17]:
text_response=llm.invoke("what is llm?")

In [18]:
text_response

AIMessage(content='LLM stands for Master of Laws, which is a postgraduate law degree that allows individuals to specialize in a specific area of law after completing their undergraduate studies in law. It is typically pursued by individuals who already have a law degree and wish to further their knowledge and expertise in a particular area of law.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 12, 'total_tokens': 73, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CZV4pg4dkl02lTyiBMiXYWcjyBgGv', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--b9f1e422-129d-4b30-91df-9c2ae859f8ba-0', usage_metadata={'input_tokens': 12, 'out

In [19]:
from langchain.chat_models.base import init_chat_model
llm=init_chat_model("openai:gpt-3.5-turbo")
llm.invoke("what is ml?")

AIMessage(content='ML stands for Machine Learning, which is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that allow computers to learn and improve from experience without being explicitly programmed. Machine learning algorithms can analyze and identify patterns in data to make predictions or decisions without human intervention.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 11, 'total_tokens': 67, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CZV4sD2f8hqEfYkhQXJPWwIR798Lo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--d7501380-31ea-4aeb-ae42-f7b9261bb635-0',

# Modern Rag Chain

In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableMap, RunnableLambda
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain.agents.middleware import dynamic_prompt, ModelRequest

In [21]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [22]:
retriever=vectorstore.as_retriever(
    search_kwargs={"k":3}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000210D0B28340>, search_kwargs={'k': 3})

In [23]:
system_prompt = """
    YOu are an assistant for question-answer tasks.
    Use the following pieces of retrieved context to answer the question.
    If you don't know the answer, just say that you don't know.
    Use three sentences max and keep the answer concise.

    Context: {context}
"""

prompt=ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [24]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\n    YOu are an assistant for question-answer tasks.\n    Use the following pieces of retrieved context to answer the question.\n    If you don't know the answer, just say that you don't know.\n    Use three sentences max and keep the answer concise.\n\n    Context: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [25]:
# create a doc chain
chain = prompt | llm


In [26]:
chain

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\n    YOu are an assistant for question-answer tasks.\n    Use the following pieces of retrieved context to answer the question.\n    If you don't know the answer, just say that you don't know.\n    Use three sentences max and keep the answer concise.\n\n    Context: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000021085E187F0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000021085E1B460>, root_client=<openai.OpenAI object at 0x0000021085E1B040>, root_async_client=<openai.A

In [38]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs) if docs else "No relevant context found."

SYSTEM = """
You are an assistant for question-answer tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences max and keep the answer concise.

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "{input}")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

chain = (
    {
        # feed the user question to the retriever
        "context": itemgetter("input") | retriever | RunnableLambda(format_docs),
        # also pass the original input into the prompt
        "input": itemgetter("input"),
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = chain.invoke({"input": "Summarize the policy on reimbursements."})
print(answer)


I don't know.


In [39]:
response = chain.invoke({
    "input": "What is deep learning"
})

print(response)


Deep learning is a subset of machine learning that uses neural networks with many layers to analyze various forms of data. It is particularly effective in tasks such as image and speech recognition, natural language processing, and more. Deep learning models learn from large amounts of data to improve their accuracy over time.


In [56]:
# pip install langchain-core langchain-openai

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# === you already have this ===
# documents: list[Document] from DirectoryLoader

def format_docs(docs):
    return "\n\n".join(getattr(d, "page_content", str(d)) for d in docs) if docs else "No relevant context found."

SYSTEM = """You are an assistant for question-answer tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences max and keep the answer concise.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "{input}")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ✅ Pass the docs at invoke-time
chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["context"]))
    | prompt
    | llm
    | StrOutputParser()
)

answer = chain.invoke({
    "input": "what is deep learning?",
    "context": documents   # <-- your loaded docs go here
})

print(answer)


Deep learning is a subset of machine learning that is based on artificial neural networks, which are inspired by the human brain. It involves layers of interconnected nodes and has significantly advanced fields such as computer vision, natural language processing, and speech recognition. Deep learning techniques, like Convolutional Neural Networks (CNNs) and Recurrent Neural Networks (RNNs), are particularly effective for processing images and sequential data, respectively.


In [57]:
for d in documents:
    print(getattr(d,"page_content",str(d)))


    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    

    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image proce